# Qwen2.5-3B BASE + RL — DAPO, 300 updates

Roadmap **B-RL**. Starts from **Qwen/Qwen2.5-3B**, revision
`3aab1f1954e9cc14eb9509a215f9e5ca08227a9b`, the same base checkpoint as B0.
Full-parameter RL from the original base weights. No SFT checkpoint is loaded.

Matches the instruct + RL experiment's DAPO data and optimization settings:
- Verified 14,068-row English DAPO dataset; dataset commit is recorded when loaded.
- G=8, 16 retained mixed-correctness questions / 128 responses per rollout.
- Two disjoint 64-response updates per rollout; each retained response is used once.
- **300 updates**, 150 fresh rollout batches, checkpoints every **20 updates**.
- LR **1e-6**, 20-update warmup then constant; sampling temperature/top-p **1**.
- Rule-based +1/-1 reward, clip 0.20/0.28, group normalization, token-level loss, no KL.
- Response cap **20,480 tokens**; soft length penalty above **16,384**; no budget forcing.

Uses the base tokenizer's shipped chat template/default system message and the same
English math instruction. Preserves the original base **EOS `<|endoftext|>` (151643)**
and original padding token (also `<|endoftext|>`). Generation stops on the native EOS;
no EOS override, EOS suppression, extra stop tokens, or vocabulary additions.
Completion masks are based on actual response lengths, so generated EOS receives loss
while added padding does not, even though EOS and padding share the same token ID.

**RTX PRO 6000 Blackwell 96GB**, FP32 parameters/Adam state, BF16 compute/rollouts,
gradient checkpointing, and microbatch 1. Includes the Qwen memory fix: checkpointed
vocabulary projection in 128-token chunks. vLLM sleeps during optimization.
Fifteen full resumable checkpoints plus smoke need roughly **650 GB** of Drive space.

Code/config/locks are embedded. Use a fresh GPU runtime, run setup and preparation,
then enable the separate smoke and full-training cells. Training is independent of
the completed instruct + RL and Kimi SFT runs. No evaluation or HF upload is performed.
Across 300 updates: 2,400 retained question selections (not necessarily unique).
Dynamic sampling can inspect additional questions to find mixed-correctness groups.

**Updated EOS behavior:** stop the old Colab run and start this notebook in a fresh
runtime. The default output is now `base-rl-native-eos`, separate from the earlier
`base-rl` attempt. Start again from the original base weights; do not resume that
attempt. Its files are preserved. Check smoke response lengths before full training;
correcting EOS does not guarantee short responses or establish the earlier slowdown's cause.

In [ ]:
OUTPUT_ROOT = "/content/drive/MyDrive/LG-AIME-Qwen25-3B-Experiments/base-rl-native-eos"
CODE_PARENT = "/content/lg-aime-dapo-qwen25-base"
RL_ENV = "/content/lg-dapo-base-env"
WORK_DIR = "/content/qwen25-base-dapo-work"
CONFIG = "configs/dapo_qwen25_3b_base.yaml"

## Setup

Enable `HF_TOKEN` in Colab Secrets. The pinned Qwen base model and verified dataset
are loaded from Hugging Face. The embedded code is extracted into a versioned local
folder with a reproducible Git snapshot for strict resume checks; no GitHub clone.

In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.environ["HF_TOKEN"]:
    raise ValueError("Enable HF_TOKEN in Colab Secrets")
BUNDLE = '...'
bundle_bytes = base64.b64decode(BUNDLE)
bundle_id = hashlib.sha256(bundle_bytes).hexdigest()
CODE_ROOT = str(Path(CODE_PARENT) / bundle_id[:16])
Path(CODE_ROOT).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as bundle:
    for name in bundle.namelist():
        path = Path(CODE_ROOT) / name
        data = bundle.read(name)
        if path.exists() and path.read_bytes() != data:
            raise ValueError(f"Bundled file changed: {path}. Use a fresh CODE_PARENT.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
def git(*args, **kwargs):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True, **kwargs).strip()
if not (Path(CODE_ROOT) / ".git").exists():
    git("init", "--quiet")
    git("config", "core.autocrlf", "false")
    git("config", "core.filemode", "false")
    git("add", ".")
    commit_env = dict(os.environ, GIT_AUTHOR_NAME="Notebook bundle", GIT_COMMITTER_NAME="Notebook bundle",
                      GIT_AUTHOR_EMAIL="bundle@localhost", GIT_COMMITTER_EMAIL="bundle@localhost",
                      GIT_AUTHOR_DATE="2026-01-01T00:00:00+00:00", GIT_COMMITTER_DATE="2026-01-01T00:00:00+00:00")
    git("-c", "commit.gpgsign=false", "commit", "--quiet", "-m", "Bundled DAPO " + bundle_id,
        env=commit_env)
if git("status", "--porcelain"):
    raise ValueError("Bundled code was modified; use a fresh CODE_PARENT.")
print("Bundled code:", CODE_ROOT, "commit:", git("rev-parse", "HEAD"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_dapo_runtime.py", "--venv", RL_ENV,
                       "--architecture", "Qwen2ForCausalLM"], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged

## Paths and training command

Dedicated base + RL checkpoints and logs. The runner's `--model-kind base` means
starting from the HF model specified by this config: **Qwen2.5-3B base**.

In [ ]:
from train.dapo_data import load_config, batch_schedule
cfg = load_config(Path(CODE_ROOT) / CONFIG)
if cfg["model"]["repo"] != "Qwen/Qwen2.5-3B":
    raise ValueError("Use the Qwen2.5-3B base config")
RUN_NAME = cfg["run_name_prefix"] + "_base"
run_logs = Path(OUTPUT_ROOT) / "logs" / RUN_NAME
checkpoint_root = Path(OUTPUT_ROOT) / "checkpoints" / RUN_NAME
TRAIN_COMMAND = [str(Path(RL_ENV) / "bin/python"), "train/run_dapo.py", "--config", CONFIG,
                 "--model-kind", "base", "--output-root", OUTPUT_ROOT, "--work-dir", WORK_DIR]
print("Starting model:", cfg["model"]["repo"])
print("Updates per rollout:", batch_schedule(cfg)["updates_per_rollout"])
print("Checkpoints:", checkpoint_root)
print("Training logs:", run_logs)

## Prepare and inspect the prompt
Checks dataset provenance, tokenizer access, native stop/padding tokens and prompt lengths.
Does not load model weights. Saves resolved configuration and the data report to Drive.

In [ ]:
run_logged(TRAIN_COMMAND + ["--prepare-only"], cwd=CODE_ROOT,
           log_path=run_logs / "prepare_console.log")

## GPU smoke — two updates on one rollout
Uses G=8, two retained mixed groups (16 responses), two disjoint 8-response minibatches,
the full token cap, and no warmup.
Verifies generation, rewards, minibatch updates and checkpoint saving on your GPU.
Smoke has its own timestamped directory; its weights are never used for the full run.
The rows must have minibatch_index 1 then 2, with policy_iteration=1 in both.
The second row has cached_rollout=true, reused_rollout=false and new_generated_tokens=0.
Clipping can still be zero if the update is small. A nonzero clip fraction is not required.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    from datetime import datetime, timezone
    smoke_root = Path(OUTPUT_ROOT) / "smoke_attempts" / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
    command = list(TRAIN_COMMAND)
    command[command.index("--output-root") + 1] = str(smoke_root)
    run_logged(command + ["--smoke"], cwd=CODE_ROOT,
               log_path=smoke_root / "smoke_console.log", compact_progress=True)
    smoke_log = smoke_root / "logs" / (RUN_NAME + "_smoke") / "steps.jsonl"
    records = [json.loads(line) for line in smoke_log.read_text().splitlines()]
    assert [r["minibatch_index"] for r in records] == [1, 2]
    assert [r["policy_iteration"] for r in records] == [1, 1]
    assert not any(r["reused_rollout"] for r in records)
    assert records[-1]["cached_rollout"] and records[-1]["new_generated_tokens"] == 0
    print("Two-update smoke completed. Logs:", smoke_log)
else:
    print("Set RUN_SMOKE = True to run the separate GPU smoke.")

## Full training — 300 total optimizer updates

Set `RUN_TRAINING = True`. The first run starts from the pinned Qwen base model.
Saves checkpoints 20/40/.../300 with model, tokenizer, optimizer, scheduler and RNG.
Dynamic sampling may generate extra groups to find 16 mixed-correctness questions.

On rerun, the latest completed checkpoint in this run is selected automatically.
`RESUME_CHECKPOINT` can explicitly select that same latest checkpoint. Resume restores
training state and starts a fresh rollout at a complete cycle boundary. Work after
the latest saved checkpoint is replayed; abandoned logs are archived by the runner.
After a disconnect, rerun setup/preparation in a fresh GPU runtime with the same notebook.

This run has no recovery parent. If it fails before checkpoint 20, no saved optimizer
state exists; choose a new `OUTPUT_ROOT` to restart and preserve the failure logs.
Keep code/config/runtime unchanged for resume. A completed checkpoint-300 run is skipped.

In [ ]:
RUN_TRAINING = False
RESUME_CHECKPOINT = ""
if RUN_TRAINING:
    from train.dapo_resume import check_checkpoint
    command = list(TRAIN_COMMAND)
    complete = sorted(
        (p for p in checkpoint_root.glob("checkpoint-*") if (p / "dapo_checkpoint.json").is_file()),
        key=lambda p: int(p.name.split("-")[-1]))
    latest = RESUME_CHECKPOINT or (str(complete[-1]) if complete else "")
    done = False
    if latest:
        if not complete or Path(latest).resolve() != complete[-1].resolve():
            raise ValueError("Resume only the latest completed checkpoint of this base + RL run")
        marker, state = check_checkpoint(latest)
        previous = json.loads((run_logs / "run_manifest.json").read_text())
        if (marker["run_identity"] != previous["identity"]
                or previous["config"]["model"] != cfg["model"]
                or previous["config"]["run_name_prefix"] != cfg["run_name_prefix"]
                or previous["config"].get("tokenizer") != cfg.get("tokenizer")
                or previous["config"].get("generation") != cfg.get("generation")):
            raise ValueError("Checkpoint belongs to another training run")
        command += ["--resume-from-checkpoint", latest]
        done = marker["global_step"] == cfg["training"]["max_steps"]
    if done:
        print("The 300-update base + RL run is already complete:", latest)
    else:
        print("Start/resume from:", latest or cfg["model"]["repo"])
        run_logged(command, cwd=CODE_ROOT, log_path=run_logs / "training_console.log",
                   compact_progress=True)
else:
    print("Set RUN_TRAINING = True to start/resume the base + RL run.")

## Drive outputs
Under OUTPUT_ROOT:
- `checkpoints/dapo_qwen25_3b_pretrained_base/checkpoint-<step>/`: full resumable checkpoints.
- `logs/dapo_qwen25_3b_pretrained_base/steps.jsonl`: loss, gradient norm, learning rate, reward,
  entropy/clipping, candidate accuracy, acceptance counts, length and timing metrics.
- `logs/dapo_qwen25_3b_pretrained_base/rollouts/attempt_*/update_*/`: raw answers and scoring records.
- `logs/dapo_qwen25_3b_pretrained_base/`: console logs, resolved config, data report and run manifest.
- `smoke_attempts/<timestamp>/`: separate smoke logs and checkpoint.

`rollout_first_update`, `minibatch_index` and `policy_iteration` identify each update. Rollout statistics
repeat on the second minibatch; sum `new_generated_tokens` for actual new generation.
Cached rollout accuracy is not a fresh measurement of the updated model.
Console logs are automatic. Save the notebook itself in Drive through Colab if you
also want its displayed cell outputs. No checkpoint weights are packaged for sharing.

In [ ]:
print("Final model:", checkpoint_root / "checkpoint-300")
print("Loss / reward / length history:", run_logs / "steps.jsonl")
print("Run manifest:", run_logs / "run_manifest.json")
print("Console log:", run_logs / "training_console.log")